# nb03 supplementary: TabPFN v2 baseline

This notebook is a thin wrapper around `scripts/opx_tb_nb03_tabpfn_baseline.py`. Its purpose is to add the TabPFN v2 foundation model (Hollmann et al. 2025, Nature 637) as a supplementary baseline against the pre-registered v10 tuned pipeline. No Optuna tuning, no feature-set sweep, no stacking, no SHAP. Raw features, 5 seeds (42-46), CPU inference. See `docs/nb03_tabpfn_plan.md` for the methods justification.

**Run the baseline from the separate tabpfn venv:**
```
python -m venv .venv-tabpfn
.venv-tabpfn\Scripts\activate
pip install -r requirements-tabpfn.txt
.venv-tabpfn\Scripts\python.exe scripts/opx_tb_nb03_tabpfn_baseline.py --pipeline all
```

In [ ]:
# Run the baseline. Comment this cell out after the first successful run;
# it reuses checkpoints in results/v10_tabpfn_checkpoints/ on re-entry.
import subprocess, sys
# Use the tabpfn venv explicitly; the kernel running this notebook must NOT be
# the tabpfn venv (we want the main venv for inspection).
cmd = ['.venv-tabpfn/Scripts/python.exe',
       'scripts/opx_tb_nb03_tabpfn_baseline.py', '--pipeline', 'all']
print('running:', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:], file=sys.stderr)
    raise SystemExit(r.returncode)

In [ ]:
# Display TabPFN summary and head-to-head against the v10 aggregate-best
# tuned model per (pipeline, track, target).
import pandas as pd
from pathlib import Path
from config import RESULTS

tab = pd.read_csv(RESULTS / 'v10_tabpfn_multiseed_summary.csv')
print('TabPFN summary:')
print(tab.to_string(index=False))

opx = pd.read_csv(RESULTS / 'v10_opx_multiseed_summary.csv')
cpx = pd.read_csv(RESULTS / 'v10_cpx_multiseed_summary.csv')
v10 = pd.concat([opx, cpx], ignore_index=True)

# Aggregate-best v10 model per cell (smallest mean RMSE across tuned models).
best_v10 = (v10.sort_values('mean')
               .groupby(['track', 'target'])
               .first()
               .reset_index()[['track', 'target', 'model', 'feature_set',
                                'mean', 'std']]
               .rename(columns={'mean': 'v10_best_rmse',
                                'std':  'v10_best_std',
                                'model': 'v10_best_model',
                                'feature_set': 'v10_best_fs'}))

tab_small = tab[['track', 'target', 'mean', 'std']].rename(
    columns={'mean': 'tabpfn_rmse', 'std': 'tabpfn_std'})

h2h = best_v10.merge(tab_small, on=['track', 'target'], how='inner')
h2h['delta_rmse'] = h2h['tabpfn_rmse'] - h2h['v10_best_rmse']
print('\nHead-to-head (TabPFN vs v10 best tuned):')
print(h2h.to_string(index=False))

Outputs feed into `nb04_v10_benchmark` (TabPFN row added to benchmark figure/table), `nb05_loso_validation` (optional extension, not run by default here), and `nbF_figures` (new fig35_tabpfn_vs_v10 if TabPFN is competitive or superior). See `docs/nb03_tabpfn_plan.md` for methods justification. TabPFN is deliberately excluded from `V10_BASE_ORDER` and the pre-registered model roster; it is a post-hoc supplementary benchmark.